# 분기하며 Tool 을 사용하는 랭그래프

In [ ]:
# 1. 랭체인 설치
%pip install langchain

# 2. 랭체인에서 OpenAI를 사용하기 위한 패키지
%pip install langchain_openai

# 3. 랭그래프 설치
%pip install langgraph

# 2. 랭체인에서 OpenAI를 사용하기 위한 패키지
%pip install langchain_openai

# 3. 랭그래프 설치
%pip install langgraph

In [ ]:
import json
import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

load_dotenv()
MODEL = ChatOpenAI(model="gpt-4o-mini")

c:\Users\sycsn\OneDrive\STUDY\Coding\PYTHON\PY_LLM\MYVENV\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
class State(TypedDict):
    user_input: str
    messages: Annotated[list, add_messages]
    
graph_builder = StateGraph(State)

### TOOL 정의
실제로 일을 수행하는 TOOL 을 정의 한다.
 - LLM 이 '특정TOOL을 사용하겠다는 판단'을 하고 Tool Call 만 한다.
 - 실제 실행은 뒤에서 만들 BasicToolNode 가 담당
 - LLM 의 Tool 사용 판단 기준
      1. 설명(docstring)
      2. 함수 이름
      3. 파라미터 이름과 타입과 순서

### 바인딩할 TOOL 목록
모델이 tool call 대상으로 인식할 수 있는 전체 TOOL LIST 생성

### bind_tools : 모델에 TOOL 바인딩
모델에게 "이런 도구들이 있다"라고 알려주는 과정
이 설정을 해야 모델이 tool_calls를 포함한 응답을 생성할 수 있다.

### 시스템 프롬프트 : Tool 사용 시점 지시

### BasicToolNode
AIMessage 에서 TOOL 호출 요청이 있을 때 이를 실행시키는 역할

# __init__ : tools by name : key[tool 이름], value[실제 tool 객체] 로 하는 dict. 를 생성하는 초기화 단계
{
    "add_numbers": <tool object>,
    "multiply_numbers": <tool object>,
    "reverse_text": <tool object>,
}
이렇게 저장해두면, 모델이 "add_numbers"를 호출하라고 했을 때
당 함수를 빠르게 찾아 실행할 수 있다.

# __call__ : 호출 메서드 - 클래스를 함수처럼 사용하게 해준다.
클래스를 Langgraph 노드처럼 동작하게 한다.
- Tool 사용 요청의 경우 마지막 메세지의 형식은
AIMessage(... tool_calls=[...])






In [ ]:
# BasicToolNode
class BasicToolNode:
    """마지막 AIMessage의 tool_calls를 읽어서 실제 tool을 실행하는 노드"""

    # tools_by_name - 전체 Tools Dict 생성
    def __init__(self, tools:list) -> None:
        self.tools_by_name = {
            tool.name: tool for tool in tools
        }

    # 노드처럼 동작하는 Call 호출 메서드
    def __call__(self, state: dict):
        # 1. messages 읽기
        messages = state.get("messgaes", [])
        last_message = messages[-1] #AIMessage(... tool_calls=[...])

        # 2. 요청 Tool 순회
        ToolResults = []
        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call

In [ ]:
# Tool 객체로 함수 정의
@tool
def add_numbers(number_a: int, number_b: int) -> str:
    # docstring
    """
    두 정수를 입력받아 합을 계산합니다.

    사용 시점:
        - 사용자가 두 수의 합이나 덧셈 결과를 요청할 때 사용합니다.

    입력값:
        - number_a: 첫 번째 정수
        - number_b: 두 번째 정수

    반환값:
        - "a + b = result" 형식의 문자열
    """

    return f"{number_a} + {number_b} = {number_a + number_b}"

@tool
def multiply_numbers(number_a: int, number_b:int) -> str:
    """
    두 정수를 곱한 결과를 반환한다.
    사용시점 : 사용자가 두수의 곱을 요청할 때 사용
    입력값: number_a, nubmer_b : 첫,두번째 정수
    반환값: "{a} x {b} = {a * b}" 형식의 문자열
    """

    return f"{number_a} x {number_b} = {number_a * number_b}"

@tool
def reverse_text(text: str) -> str:
    """
    입력 문자열을 뒤집어서 반환한다.
    """

    return text[::-1]

# TOOLS: tool 객체들 list
TOOLS = [
    add_numbers,
    multiply_numbers,
    reverse_text
]

# TOOLS 바인딩
MODEL = MODEL.bind_tools(TOOLS)

# 시스템 프롬프트 : TOOL 사용 시점을 알려줄 지시
SYSTEM_PROMPT = """
너는 도우미다.
사용자 요청을 보고 계산이나 문자열 뒤집기가 필요하면 반드시 적절한 도구를 호출하라.
최종 답변을 한국어 한두 문장으로 간단히 정리하라.
""".strip()



### NODE
1. init_system : 첫 턴에만 SystemMessage를 State 에 추가, 이후 턴에는 중복 삽입을 막음
2. add_user_message : 이번 턴의 user_input을 HumanMessage로 변환해 추가
3. generate : 준비된 state["messages"]만 그대로 invoke(...)

### 대화 턴 진행에 따른 State["messages"]
[
    SystemMessage(BASE), #1st Turn
    HumanMessage(...),   #1st
    AIMessage(...),      #1st

    HumanMessage(...),   #2nd
    AIMessage(...),      #2nd

    HumanMessage(...),   #3rd
    AIMessage(...),      #3rd
]


In [ ]:
def NODE_init_system(state: State):
    """첫 턴에만 SystemMessage를 messages 맨 앞에 추가한다."""
    messages = state.get("messages", []) #massages 없으면 빈 리스트 get

    # 메세지가 이미 있고, 첫 메세지가 시스템 메세지 객체면 Node 스킵
    if messages and isinstance(messages[0], SystemMessage):
        return {}
    
    else:
        return {
            "messages" : [
                SystemMessage(content=SYSTEM_PROMPT)
            ]
        }
    
def NODE_add_user_message(state: State):
    """이번 턴의 사용자 입력을 HumanMessage로 변환해 messages에 추가한다."""
    
    return {
        "messages" : [
            HumanMessage(content = state["user_input"])
        ]
    }

def NODE_generate(state: State):
    """현재 상태(messages)를 바탕으로 다음 AIMessage를 생성한다."""
    
    response = MODEL.invoke(state["messages"])
    return {"messages": [response]}


